# Урок 20 — Контекстні Менеджери: Архітектура Гарантованого Життєвого Циклу

**Модуль 2 · Python Intermediate · Автор: Viktor Nikoriak**

---

## Про що цей урок?

Як розробники, ми маємо виходити з простого припущення: **будь-яка зовнішня операція колись завершиться помилкою**.
Мережеві з'єднання обриваються. Бази даних відхиляють запити. Розробники передають невалідні дані.

Якщо ви покладаєтесь на ручне звільнення ресурсів — виклик `.close()`, `.release()` або `.rollback()` —  
ви неминуче отримаєте **«помилки опущення» (errors of omission)**: баги, де розробник просто *забуває* прибрати за собою.

**Контекстний менеджер — це відповідь Python на цю проблему.**  
Він перетворює ресурс на строгу **просторову межу (scope)**:  
потік входить у межу — ресурс захоплюється; потік виходить — ресурс гарантовано вивільняється.

| Частина | Патерн | Ключова ідея |
|---------|--------|--------------|
| 1 | `try...finally` → `with` | Еволюція управління ресурсами |
| 2 | Протокол: `__enter__` / `__exit__` | Механіка «під капотом» |
| 3 | Клас-менеджер: `ListTransaction` | Паттерн транзакції |
| 4 | Клас-менеджер: `RedirectStdout` | Паттерн тимчасового стану |
| 5 | `@contextmanager` | Генератори як менеджери контексту |
| 6 | Lock Pattern | Уникнення взаємного блокування |
| 7 | Transaction Pattern | «Все або нічого» |
| 8 | `__exit__` як «вертухай» | Управління та придушення винятків |
| 9 | `contextlib.suppress` | Стандартна бібліотека у дії |

---

## Частина 1: Від `try...finally` до `with` — Еволюція управління ресурсами

---

### Передбачення

> **Питання:** Ви відкрили файл і отримали виняток посередині запису.  
> Файл закрився?
> Скільки разів таке може трапитись, перш ніж сервер впаде з `OSError: Too many open files`?

### Пояснення: Крихкість ручного управління ресурсами

У традиційному підході без контекстних менеджерів управління ресурсами є **багатослівним і крихким**.  
Програмісти використовують блок `try...finally`:

```python
f = open('hello.txt', 'w')
try:
    f.write('привіт, світ!')
finally:
    f.close()  # Виконається ЗАВЖДИ — навіть якщо write() впав
```

Цей підхід **покладається на дисципліну розробника**.  
Якщо хтось забуде `try...finally`, виникне витік дескрипторів файлів або «зависання» БД.

Інструкція `with` замінює цей шаблон, **інкапсулюючи** його у саму структуру об'єкта:

```python
with open('hello.txt', 'w') as f:
    f.write('привіт, світ!')
# Файл гарантовано закрито тут, навіть якщо write() впав
```

> **Архітектурна думка:**  
> `with` створює чітку **просторову межу** (блок коду).  
> Вихід з межі = гарантоване очищення. Завжди.

In [ ]:
# Демонстрація: чому ручне close() ненадійне

# --- Небезпечний код ---
f = open('demo.txt', 'w')
try:
    f.write('рядок 1\n')
    # Уявімо: тут стався виняток
    # Без finally — f.close() ніколи не викличеться!
finally:
    f.close()
    print('Файл закрито вручну (try/finally)')

print('---')

# --- Безпечний код через with ---
with open('demo.txt', 'w') as f:
    f.write('рядок 1\n')
    f.write('рядок 2\n')
# Тут Python автоматично викликав f.__exit__() → f.close()

print(f'Файл закрито? {f.closed}')  # True — гарантовано

## Частина 2: Протокол `__enter__` та `__exit__` — Механіка «під капотом»

---

### Пояснення: Два дандер-методи, що утворюють протокол

Щоб об'єкт міг працювати з `with`, йому **не потрібно успадковувати жоден базовий клас**.  
Python використовує качину типізацію: достатньо реалізувати два магічних методи.

| Метод | Коли викликається | Що робить |
|-------|------------------|-----------|
| `__enter__(self)` | При **вході** в блок `with` | Захоплює ресурс, повертає об'єкт для `as` |
| `__exit__(self, exc_type, exc_val, exc_tb)` | При **виході** з блоку `with` | Звільняє ресурс, може обробити виняток |

**Покрокова анатомія виконання `with expression as var:`**

1. **Оцінка виразу** — Python обчислює `expression`, отримуючи об'єкт менеджера.
2. **Вхід у контекст** — викликається `__enter__()`.
3. **Прив'язка змінної** — значення, яке повернув `__enter__()`, присвоюється `var`.
   > *Нюанс:* `__enter__` не зобов'язаний повертати `self` — він може повернути будь-який допоміжний об'єкт.
4. **Виконання тіла** — виконується ваш код усередині блоку.
5. **Вихід з контексту** — незалежно від того, як завершився блок (успішно, `return`, `break`, виняток), викликається `__exit__()`.
6. **Обробка винятку** — якщо у блоці стався виняток, `__exit__` отримує його деталі.  
   Якщо `__exit__` повертає `True` — виняток пригнічується.  
   Якщо `False` або `None` — виняток розповсюджується далі.

```
with cm_object as resource:
    ┌──────────────────────────────────┐
    │  cm_object.__enter__() викликано │  ← setup
    │  resource = результат __enter__  │
    │  ... ваш код ...                 │
    │  cm_object.__exit__(...)викликано│  ← teardown (ЗАВЖДИ)
    └──────────────────────────────────┘
```

In [ ]:
# Мінімальний клас-менеджер контексту — «оголена механіка»

class SimpleContext:
    """Демонструє послідовність викликів __enter__ та __exit__."""

    def __enter__(self):
        # Фаза SETUP: тут захоплюємо ресурс
        print('__enter__: входимо в контекст')
        return self  # Повертаємо self → присвоюється змінній після 'as'

    def __exit__(self, exc_type, exc_val, exc_tb):
        # Фаза TEARDOWN: тут звільняємо ресурс
        # exc_type, exc_val, exc_tb = None, None, None — якщо помилок не було
        print(f'__exit__: виходимо з контексту')
        print(f'  exc_type={exc_type}, exc_val={exc_val}')
        return False  # Не пригнічуємо виняток — нехай розповсюджується


print('=== Нормальне завершення ===')
with SimpleContext() as ctx:
    print(f'  всередині блоку, ctx={ctx}')

print()
print('=== Завершення з винятком ===')
try:
    with SimpleContext() as ctx:
        print('  робимо щось...')
        raise ValueError('щось пішло не так!')
except ValueError as e:
    print(f'  виняток перехоплено зовні: {e}')

## Частина 3: Паттерн Транзакції — `ListTransaction`

---

### Пояснення: «Все або нічого» — Unit of Work

Уявіть: ви модифікуєте список або базу даних.  
Якщо під час модифікації стається помилка, ви отримаєте **частково оновлені, пошкоджені дані**.

Паттерн **Транзакція (Transaction / Unit of Work)** вирішує це через механіку **«Commit / Rollback»**:
- Усі зміни вносяться у **робочу копію** (workingcopy), а не в оригінал.
- `__enter__` → створює робочу копію і повертає її.
- `__exit__` → якщо помилок **не було** (`exc_type is None`) → **Commit**: копіюємо назад в оригінал.  
  Якщо помилка **була** → **Rollback**: просто нічого не робимо, копія знищується.

Це **той самий архітектурний патерн**, який використовують SQLAlchemy та `sqlite3`.

In [ ]:
class ListTransaction:
    """Контекстний менеджер, що реалізує паттерн Транзакції для списку."""

    def __init__(self, thelist):
        self.thelist = thelist  # Оригінальний список — захищений

    def __enter__(self):
        # SETUP: створюємо ізольовану робочу копію
        # Клієнт отримує КОПІЮ, а не сам оригінал
        self.workingcopy = list(self.thelist)
        return self.workingcopy

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is None:
            # Успіх → COMMIT: замінюємо вміст оригіналу вмістом копії
            # list[:] = ... модифікує оригінальний об'єкт на місці (in-place)
            self.thelist[:] = self.workingcopy
            print('COMMIT: зміни зафіксовано')
        else:
            # Помилка → ROLLBACK: просто не робимо нічого
            # Робоча копія просто знищиться збирачем сміття
            print(f'ROLLBACK: помилка {exc_type.__name__} — оригінал збережено')

        # Повертаємо False → не пригнічуємо виняток
        return False

In [ ]:
# === Сценарій 1: Успішна транзакція (Commit) ===
items = [1, 2, 3]
print(f'До: {items}')

with ListTransaction(items) as working:
    working.append(4)
    working.append(5)
    # Помилок не було → __exit__ отримає exc_type=None → Commit

print(f'Після: {items}')  # [1, 2, 3, 4, 5]

print()

# === Сценарій 2: Транзакція з помилкою (Rollback) ===
items = [1, 2, 3]
print(f'До: {items}')

try:
    with ListTransaction(items) as working:
        working.append(4)
        working.append(5)
        raise RuntimeError('Раптова помилка мережі!')  # Імітуємо збій
except RuntimeError:
    pass

# Зміни відкотились — список залишився незмінним
print(f'Після: {items}')  # [1, 2, 3] — захищено!

## Частина 4: Паттерн Тимчасового Стану — `RedirectStdout`

---

### Пояснення: Захист глобального стану

Іноді системі потрібно **тимчасово змінити глобальний стан** — рівень логування, точність обчислень,  
стандартний потік виводу — виконати роботу, а потім **гарантовано повернути все як було**.

Без контекстного менеджера це небезпечно: якщо код між «змінити» і «відновити» впаде —  
система залишиться в пошкодженому стані назавжди.

Схема паттерну:
- `__enter__` → зберігає **старий стан**, встановлює **новий стан**.
- `__exit__` → **гарантовано відновлює** старий стан (навіть при помилці).

> *Реальні приклади зі стандартної бібліотеки:*  
> `contextlib.redirect_stdout`, `decimal.localcontext()`, `unittest.mock.patch`.

In [ ]:
import sys
import io


class RedirectStdout:
    """Тимчасово перенаправляє sys.stdout на інший потік."""

    def __init__(self, new_target):
        self.new_target = new_target  # Новий потік виводу

    def __enter__(self):
        # SETUP: зберігаємо поточний stdout і замінюємо його
        self.original_stdout = sys.stdout  # Зберігаємо старий стан
        sys.stdout = self.new_target       # Встановлюємо новий стан
        return self.new_target

    def __exit__(self, exc_type, exc_val, exc_tb):
        # TEARDOWN: гарантовано відновлюємо оригінальний stdout
        # Навіть якщо всередині блоку стався виняток!
        sys.stdout = self.original_stdout
        return False

In [ ]:
# Перехоплюємо вивід у рядок замість терміналу
buffer = io.StringIO()

with RedirectStdout(buffer):
    print('Це піде в буфер, а не в термінал')
    print('І це теж')

# Після виходу з блоку — stdout відновлено
print('А це вже знову в термінал!')

# Отримуємо перехоплений текст
captured = buffer.getvalue()
print(f'Вміст буфера: {repr(captured)}')

# Python сам має contextlib.redirect_stdout — покажемо для порівняння
from contextlib import redirect_stdout
buf2 = io.StringIO()
with redirect_stdout(buf2):
    print('Стандартна бібліотека робить те саме!')
print(f'buf2: {repr(buf2.getvalue())}')

## Частина 5: `@contextmanager` — Елегантність на основі генераторів

---

### Пояснення: Як `yield` ламає час і простір у функції

Створення цілого класу з `__enter__` та `__exit__` для кожної дрібної задачі порушує принцип лаконічності.  
Модуль `contextlib` та декоратор `@contextmanager` перетворюють **функцію-генератор** на контекстний менеджер.

Ключове слово `yield` **розрізає функцію навпіл**:

```
@contextmanager
def managed_resource():
    ┌─────────────────────────────────────┐
    │  код до yield  ≡  __enter__         │  ← захоплення ресурсу
    │  yield value   ≡  передача в 'as'  │  ← функція «заморожується»
    │  код після yield ≡ __exit__         │  ← звільнення ресурсу
    └─────────────────────────────────────┘
```

**Архітектурне правило: Пастка винятків**  
Якщо всередині блоку `with` виникне виняток, інтерпретатор **повторно згенерує його прямо в рядку з `yield`**.  
Якщо ви не обробите це — код після `yield` (ваш teardown) **ніколи не виконається**.  
Тому `try/finally` навколо `yield` є **архітектурним обов'язком**.

In [ ]:
import time
from contextlib import contextmanager


@contextmanager
def timethis(label):
    """Вимірює час виконання блоку коду."""
    # === Аналог __enter__ ===
    start = time.time()  # Фіксуємо час початку
    try:
        yield  # Передаємо управління тілу блоку with (значення None)
               # Функція «заморожується» тут, поки блок виконується
    finally:
        # === Аналог __exit__ ===
        # finally гарантує виконання навіть при помилці в блоці
        end = time.time()
        print(f'{label}: {end - start:.4f} сек')


# Використання:
with timethis('Зворотній відлік'):
    n = 5_000_000
    while n > 0:
        n -= 1

In [ ]:
from contextlib import contextmanager


@contextmanager
def managed_file(name):
    """Безпечно відкриває файл і гарантує його закриття."""
    # === __enter__: відкриваємо ресурс ===
    f = open(name, 'w', encoding='utf-8')
    try:
        yield f  # Об'єкт файлу передається в змінну 'as'
    finally:
        # === __exit__: ЗАВЖДИ закриваємо файл ===
        f.close()
        print(f'Файл "{name}" закрито гарантовано')


with managed_file('output.txt') as f:
    f.write('привіт, світ!\n')
    f.write('а тепер, бувай!\n')

In [ ]:
# === Що відбувається, коли НЕМАЄ try/finally? ===
# Демонстрація архітектурної помилки — небезпечний варіант

from contextlib import contextmanager


@contextmanager
def dangerous_managed_file(name):
    """НЕБЕЗПЕЧНО: немає try/finally — teardown не гарантований!"""
    f = open(name, 'w', encoding='utf-8')
    yield f
    # Якщо тут виникне помилка — f.close() НЕ ВИКЛИЧЕТЬСЯ!
    f.close()
    print('Цей рядок не виконається при помилці!')


try:
    with dangerous_managed_file('danger.txt') as f:
        f.write('записуємо...\n')
        raise IOError('Симуляція збою диска!')
except IOError as e:
    print(f'Помилка: {e}')
    print(f'Файл закрито? {f.closed}')  # False — витік ресурсу!

## Частина 6: Lock Pattern — Уникнення Взаємного Блокування

---

### Пояснення: Deadlock — Смерть від дисципліни

У багатопотоковому середовищі керування м'ютексами (locks) є надзвичайно небезпечним.  
Забутий виклик `lock.release()` призводить до **взаємного блокування (deadlock)**,  
що повністю зупиняє всі потоки, які очікують цей ресурс.

Використання `with lock:` **гарантує зняття блокування**, навіть якщо всередині критичної секції виник виняток.

**Просунутий рівень: Deadlock між двома блокуваннями**  
Катастрофічний deadlock виникає якщо:
- Потік A захоплює ресурс 1, чекає ресурс 2
- Потік B захоплює ресурс 2, чекає ресурс 1 → **взаємне очікування = завішування**

**Архітектурне рішення:** сортувати блокування за `id()` — це математично доводить,  
що кругового очікування бути не може.

In [ ]:
import threading


class SharedCounter:
    """Потокобезпечний лічильник через контекстний менеджер."""

    def __init__(self):
        self._value = 0
        self._lock = threading.Lock()  # М'ютекс для захисту стану

    def increment(self):
        # 'with lock' — це паттерн «Вікно ексклюзивного доступу»
        # Блокування знімається ГАРАНТОВАНО при виході з блоку
        with self._lock:
            self._value += 1
        # Тут блокування вже знято — інші потоки можуть продовжити

    @property
    def value(self):
        with self._lock:  # Навіть читання захищаємо для thread-safety
            return self._value


# Тест: 5 потоків по 1000 інкрементів = рівно 5000
counter = SharedCounter()
threads = [threading.Thread(target=lambda: [counter.increment() for _ in range(1000)])
           for _ in range(5)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f'Результат: {counter.value}')  # Завжди 5000 — race condition виключено

In [ ]:
import threading
from contextlib import contextmanager


@contextmanager
def acquire(*locks):
    """
    Захоплює кілька блокувань у строго фіксованому порядку.

    Математичне доведення відсутності deadlock:
    Якщо всі потоки захоплюють блокування в одному й тому ж порядку
    (за id об'єкта), кругове очікування стає неможливим.
    """
    # Сортуємо блокування за їх ідентифікатором в пам'яті
    # Це гарантує однаковий порядок незалежно від аргументів
    sorted_locks = sorted(locks, key=lambda x: id(x))

    acquired = []
    try:
        for lock in sorted_locks:
            lock.acquire()       # Захоплюємо по одному у суворому порядку
            acquired.append(lock)
        yield  # Передаємо управління критичній секції
    finally:
        # Звільняємо у зворотному порядку (LIFO)
        for lock in reversed(acquired):
            lock.release()


# Тест: навіть якщо потоки просять блокування в різному порядку,
# acquire() примусово впорядковує їх → deadlock неможливий
x = threading.Lock()
y = threading.Lock()

with acquire(x, y):  # Потік 1: x потім y
    with acquire(y, x):  # «Потік 2»: просить y потім x, але acquire сортує → x, y
        print('Обидва блокування захоплено без deadlock!')

## Частина 7: Transaction Pattern — E-Commerce Резервування

---

### Пояснення: Паттерн Cleanup — Запобігання помилкам опущення

Уявіть систему, де замовлення **резервує товари** на складі.  
Якщо оплата картою не пройде — ми **зобов'язані** скасувати резервування.  

Без контекстного менеджера: якщо API оплати впаде з винятком — розробник *може забути* скасувати резерв.  
З контекстним менеджером: логіка скасування **вбудована в сам об'єкт** — забути неможливо.

```python
with create_grocery_list(order, inventory) as lst:
    charge_credit_card(order)  # Якщо тут виняток — unreserve_items() викличеться автоматично
```

In [ ]:
from contextlib import contextmanager


class GroceryList:
    """Список зарезервованих товарів для замовлення."""

    def __init__(self, order_id, items):
        self.order_id = order_id
        self._reserved = []  # Зарезервовані позиції

        # Резервуємо товари при створенні
        for item in items:
            self._reserved.append(item)
            print(f'  [WAREHOUSE] Зарезервовано: {item}')

    def has_reserved_items(self):
        return bool(self._reserved)

    def unreserve_items(self):
        print(f'  [WAREHOUSE] Знімаємо резерв для замовлення {self.order_id}')
        self._reserved.clear()


@contextmanager
def create_grocery_list(order_id, items):
    """Резервує товари і гарантує скасування при помилці."""
    grocery_list = GroceryList(order_id, items)
    try:
        yield grocery_list  # Передаємо список клієнтському коду
    finally:
        # Очищення: якщо товари досі зарезервовані після виходу — скасовуємо
        if grocery_list.has_reserved_items():
            grocery_list.unreserve_items()


# === Сценарій: Оплата не пройшла ===
print('Обробка замовлення #42...')
try:
    with create_grocery_list('#42', ['яблука', 'молоко', 'хліб']) as order:
        print('  [PAYMENT] Спроба оплати...')
        raise ConnectionError('Платіжний шлюз недоступний!')  # Імітуємо збій
except ConnectionError as e:
    print(f'  [ERROR] {e}')

print('Резервування скасовано автоматично — склад вільний!')

## Частина 8: `__exit__` як «Вертухай» — Управління Винятками

---

### Пояснення: Булевий перемикач — «Пропустити» чи «Придушити»

Коли всередині блоку `with` виникає помилка, Python **не падає одразу**.  
Він ставить паузу і **передає виняток у метод `__exit__`** з таким запитом:  
*«Помилка сталася. Ти хочеш, щоб я її підняв, чи ти вже все вирішив?»*

| Аргумент `__exit__` | Значення при помилці | Значення без помилки |
|---------------------|---------------------|---------------------|
| `exc_type` | Клас винятку (`ZeroDivisionError`) | `None` |
| `exc_val` | Екземпляр винятку (з повідомленням) | `None` |
| `exc_tb` | Трасувальний об'єкт | `None` |

**Що повертає `__exit__`:**
- **`False` або `None`** → Python підіймає виняток далі по стеку (нормальна поведінка).
- **`True`** → Python вважає виняток «обробленим» і **пригнічує** його.  
  Виконання продовжується з першого рядка **після** блоку `with`.

> **Антипаттерн — «Чорна Діра»:**  
> Ніколи не пишіть `return True` без перевірки `exc_type`.  
> Інакше ви тихо ковтнете `KeyboardInterrupt`, `SystemExit` та всі інші помилки —  
> баги зникатимуть без сліду.

In [ ]:
class ErrorSuppressor:
    """Вибірково пригнічує конкретний тип винятку."""

    def __init__(self, target_exception):
        self.target = target_exception  # Який тип помилки нас цікавить

    def __enter__(self):
        print('Входимо в контекст...')
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f'Виходимо з контексту (exc_type={exc_type})')

        # Якщо помилок не було — нічого не робимо
        if exc_type is None:
            return False

        # Якщо помилка збігається з нашим цільовим типом — пригнічуємо
        if issubclass(exc_type, self.target):
            print(f'  [SUPPRESSED] {exc_type.__name__}: {exc_val}')
            return True  # ← Виняток знищено тут!

        # Усі інші помилки — пропускаємо далі
        return False


# === Тест 1: Придушення ZeroDivisionError ===
print('=== Тест 1: Ділення на нуль ===')
with ErrorSuppressor(ZeroDivisionError):
    print('Обчислення...')
    result = 10 / 0          # Виняток!
    print('Цей рядок ніколи не виконається')  # Пропущено
print('Програма продовжується після блоку — виняток знищено!\n')

# === Тест 2: Незнайомий тип — пропускаємо далі ===
print('=== Тест 2: ValueError — не придушуємо ===')
try:
    with ErrorSuppressor(ZeroDivisionError):
        raise ValueError('Невідома помилка!')
except ValueError as e:
    print(f'  ValueError впав назовні: {e}')

## Частина 9: `contextlib.suppress` — Стандартна Бібліотека в Дії

---

### Пояснення: `contextlib.suppress` — готовий «Придушувач»

Python's стандартна бібліотека реалізує той самий механізм через `contextlib.suppress`.  
Це заміна громіздким конструкціям `try: ... except: pass`:

```python
# Старий стиль
try:
    os.remove('tempfile.txt')
except FileNotFoundError:
    pass

# Новий стиль — чистіше і читабельніше
with contextlib.suppress(FileNotFoundError):
    os.remove('tempfile.txt')
```

In [ ]:
import contextlib
import os

# Видаляємо файл, якщо він є — ігноруємо якщо його немає
with contextlib.suppress(FileNotFoundError):
    os.remove('nonexistent_file.txt')  # Не впаде — помилка придушена
print('Файл видалено (або його й не було — нам байдуже)')

print()

# Можна придушувати кілька типів одразу
with contextlib.suppress(FileNotFoundError, PermissionError):
    os.remove('/root/protected_file.txt')
print('Ніяких падінь, навіть при відмові доступу')

print()

# decimal.localcontext() — тимчасова точність без зміни глобальних налаштувань
import decimal

with decimal.localcontext() as ctx:
    ctx.prec = 50  # Встановлюємо локальну точність на 50 знаків
    result = decimal.Decimal(1) / decimal.Decimal(3)
    print(f'Локальна точність (50 знаків): {result}')

# Після виходу — глобальна точність відновлена
print(f'Глобальна точність: {decimal.Decimal(1) / decimal.Decimal(3)}')

## Підсумок: Таблиця Архітектурних Паттернів

---

| Паттерн | `__enter__` | `__exit__` | Приклад |
|---------|-------------|------------|---------|
| **Resource Cleanup** | Відкрити файл / сокет | `close()` | `open()`, сокети |
| **Lock / Sync** | `lock.acquire()` | `lock.release()` | `threading.Lock` |
| **Transaction** | Створити робочу копію | Commit або Rollback | SQLAlchemy, `sqlite3` |
| **Temporary State** | Зберегти і замінити стан | Відновити старий стан | `redirect_stdout`, `decimal.localcontext` |
| **Timer / Audit** | Зафіксувати початок | Зафіксувати кінець / лог | Профілювання, аудит |
| **E-commerce Reservation** | Зарезервувати ресурс | Скасувати при збої | Склад, квитки |

---

> **Ключова думка архітектора:**  
> Контекстні менеджери — це не про файли.  
> Це фундамент для створення **надійних (robust) API**, де правильна поведінка системи —  
> очищення, відкат, звільнення ресурсів — **вбудована в саму структуру коду**  
> і не потребує ручного контролю.  
>  
> Ви створюєте інтерфейси, якими **дуже легко користуватись правильно**  
> і **майже неможливо скористатись неправильно**.